# Web maps (MapLibre + deck.gl)

The `web` tier renders pyramids data as interactive MapLibre GL JS / deck.gl layers and exports a
shareable HTML page. It needs the optional `web` extra (`pip install 'digitalearth[web]'`, or
`pixi install -e web` in this repo). All data is reprojected to lon/lat through pyramids before a layer
is built; MapLibre renders Web Mercator itself.

This notebook runs headless (no browser) — it builds layer specs and writes HTML; it does not screenshot.

In [ ]:
import tempfile
from pathlib import Path

import geopandas as gpd
from shapely.geometry import Polygon
from pyramids.dataset import Dataset

from digitalearth.web import WebMap

DATA = Path('../../../examples/data')  # notebook CWD is this folder; data lives at the repo-root examples/

## Raster over a basemap

`add_raster` reprojects the band to lon/lat, colour-maps it to a transparent-NoData PNG, and places it as
a MapLibre image source; `basemap` adds a dark tile layer beneath it.

In [ ]:
dem = Dataset.read_file(str(DATA / 'acc4000.tif'))
raster_map = WebMap().add_raster(dem, cmap='viridis', opacity=0.85).basemap('CartoDark')
print('raster layers:', len(raster_map.layers))
# In a live notebook, evaluate `raster_map` on its own line to render the interactive map.

## Choropleth with graduated symbology

`choropleth` compiles class breaks from `cleopatra.styles.classify` (pure-numpy quantiles / Fisher-Jenks —
no mapclassify) into a MapLibre data-driven `step` paint expression, and exposes the breaks on
`WebMap.last_breaks`.

In [ ]:
polys = gpd.GeoDataFrame(
    {'pop': [1.0, 5.0, 9.0, 3.0, 7.0, 2.0]},
    geometry=[Polygon([(i, 0), (i + 1, 0), (i + 0.5, 1)]) for i in range(6)],
    crs=4326,
)
choro = WebMap(center=(2.5, 0.5), zoom=6).choropleth(polys, column='pop', scheme='quantiles', k=4)
choro = choro.basemap('CartoLight').tooltip(['pop'])
print('class breaks:', choro.last_breaks)
# In a live notebook, evaluate `choro` on its own line to render the interactive choropleth.

## Export a shareable HTML page

`save` writes a standalone HTML page (the data is embedded; the `maplibre-gl` engine is CDN-referenced —
pass `offline=True` to inline it for a fully offline page).

In [ ]:
out = Path(tempfile.gettempdir()) / 'digitalearth_web_map.html'
choro.save(str(out))
print('wrote', out, '(', out.stat().st_size, 'bytes )')